# Homework 2 — Put bermudeen : algorithme de Longstaff-Schwartz

## Objectif

On programme l'algorithme de **Longstaff-Schwartz (LS)** pour estimer le prix d'un **Put bermudeen** dans un modele de Black-Scholes.

L'option peut etre exercee aux dates
\[
t_k = k/N, \quad k=0,\dots,N,
\]
avec les parametres imposes :
\[
r=0.1,\quad \sigma=0.25,\quad x_0=100,\quad K=110,\quad N=10,\quad T=1.
\]

Le payoff (deja actualise vers la date 0) est
\[
\phi_k(x) = e^{-r k/N}(K-x)_+.
\]

## Rappel de la recursion et de l'idee LS

On note \(X_k = X_{t_k}\). La fonction valeur verifie
\[
V_N(x)=\phi_N(x),
\qquad
V_k(x)=\max\big(\phi_k(x),\;C_k(x)\big),\quad k=N-1,\dots,0,
\]
ou la continuation est
\[
C_k(x)=\mathbb E\big[V_{k+1}(X_{k+1})\mid X_k=x\big].
\]

Dans LS, \(C_k\) est approchee par regression (base polynomiale) sur des trajectoires Monte Carlo, en pratique sur les trajectoires **in-the-money** (ITM).

La regle d'exercice apprise est
\[
\tau^* = \inf\{k:\phi_k(X_k)\ge \widehat C_k(X_k)\}.
\]

La contrainte demandee est satisfaite ici en affichant explicitement les fonctions estimees \(\widehat C_k\) et \(\widehat V_k(x)=\max(\phi_k(x),\widehat C_k(x))\).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def simulate_bs_paths(x0, r, sigma, T, N, n_paths, seed=123):
    """
    Simulation de trajectoires Black-Scholes sous la mesure risque-neutre.
    Retour: tableau X de taille (n_paths, N+1).
    """
    dt = T / N
    rng = np.random.default_rng(seed)

    z = rng.standard_normal((n_paths, N))
    X = np.empty((n_paths, N + 1), dtype=float)
    X[:, 0] = x0

    drift = (r - 0.5 * sigma**2) * dt
    vol = sigma * np.sqrt(dt)

    for k in range(N):
        X[:, k + 1] = X[:, k] * np.exp(drift + vol * z[:, k])

    return X


def payoff_discounted(x, k, K, r, dt):
    """
    Payoff actualise a t=0: phi_k(x)=exp(-r*k*dt)*(K-x)_+.
    """
    return np.exp(-r * k * dt) * np.maximum(K - x, 0.0)


def basis_poly2(x):
    """Base de regression [1, x, x^2]."""
    x = np.asarray(x)
    return np.column_stack([np.ones_like(x), x, x**2])


def fit_continuation_model(x_itm, y_itm):
    """
    Ajuste un modele de continuation C(x).
    - Si les x ITM sont assez riches: regression quadratique.
    - Sinon: modele constant (moyenne), utile pour eviter les cas degeneres.
    """
    if x_itm.size == 0:
        return {"kind": "const", "c": 0.0}

    # Evite une regression instable quand les x sont presque tous identiques.
    if np.unique(np.round(x_itm, 10)).size < 3:
        return {"kind": "const", "c": float(np.mean(y_itm))}

    A = basis_poly2(x_itm)
    beta, *_ = np.linalg.lstsq(A, y_itm, rcond=None)
    return {"kind": "poly2", "beta": beta}


def predict_continuation(model, x):
    """
    Evalue C(x) pour un modele ajuste.
    On tronque a 0 pour rester coherent avec une valeur d'option non negative.
    """
    x = np.asarray(x)

    if model["kind"] == "poly2":
        y = basis_poly2(x) @ model["beta"]
    else:
        y = np.full_like(x, model["c"], dtype=float)

    return np.maximum(y, 0.0)

In [ ]:
def longstaff_schwartz_bermudan_put(x0, K, r, sigma, T, N, n_paths=40000, seed=123):
    """
    LS pour put bermudeen avec payoff deja actualise.

    Sorties:
    - price: estimation de V_0
    - models[k]: modele de continuation appris a la date k (k=0,...,N-1)
    - X: trajectoires simulees
    - tau: temps d'arret appris (indice de date d'exercice)
    - cashflow: payoff actualise final par trajectoire
    """
    dt = T / N
    X = simulate_bs_paths(x0, r, sigma, T, N, n_paths, seed=seed)

    # Initialisation a maturite
    cashflow = payoff_discounted(X[:, N], N, K, r, dt)
    tau = np.full(n_paths, N, dtype=int)

    models = [None] * (N + 1)
    models[N] = {"kind": "terminal"}

    # Backward induction
    for k in range(N - 1, -1, -1):
        immediate = payoff_discounted(X[:, k], k, K, r, dt)
        itm = immediate > 0.0

        # Cible de regression = valeur deja obtenue si on continue
        y_itm = cashflow[itm]
        x_itm = X[itm, k]

        model_k = fit_continuation_model(x_itm, y_itm)
        models[k] = model_k

        cont_all = predict_continuation(model_k, X[:, k])

        # Regle d'exercice apprise
        exercise_now = itm & (immediate >= cont_all)

        # Mise a jour du payoff realise et du temps d'arret
        cashflow = np.where(exercise_now, immediate, cashflow)
        tau = np.where(exercise_now, k, tau)

    price = float(np.mean(cashflow))

    return {
        "price": price,
        "models": models,
        "X": X,
        "tau": tau,
        "cashflow": cashflow,
        "dt": dt,
        "params": {
            "x0": x0,
            "K": K,
            "r": r,
            "sigma": sigma,
            "T": T,
            "N": N,
            "n_paths": n_paths,
            "seed": seed,
        },
    }

In [ ]:
# Parametres imposes dans l'enonce
r = 0.1
sigma = 0.25
x0 = 100
K = 110
N = 10
T = 1.0

# Taille Monte Carlo (on peut augmenter pour reduire le bruit)
n_paths = 40000
seed = 7

res = longstaff_schwartz_bermudan_put(
    x0=x0,
    K=K,
    r=r,
    sigma=sigma,
    T=T,
    N=N,
    n_paths=n_paths,
    seed=seed,
)

print(f"Estimation LS du prix du put bermudeen (V0): {res['price']:.6f}")

In [ ]:
# Petit diagnostic sur les temps d'arret appris
N = res["params"]["N"]
tau = res["tau"]

counts = np.bincount(tau, minlength=N+1)
print("Repartition des exercices (nombre de trajectoires par date k):")
for k, c in enumerate(counts):
    print(f"k={k:2d}  t={k/N: .2f}  count={c}")

In [ ]:
def continuation_on_grid(models, k, x_grid):
    """Continuation C_k(x) estimee apres apprentissage."""
    if k == len(models) - 1:
        return np.zeros_like(x_grid)
    return predict_continuation(models[k], x_grid)


def value_on_grid(models, k, x_grid, K, r, dt):
    """Valeur V_k(x)=max(phi_k(x), C_k(x)) sur une grille."""
    phi = payoff_discounted(x_grid, k, K, r, dt)
    if k == len(models) - 1:
        return phi
    C = continuation_on_grid(models, k, x_grid)
    return np.maximum(phi, C)


# Grille de prix pour affichage des fonctions
x_grid = np.linspace(40, 180, 400)
models = res["models"]
K = res["params"]["K"]
r = res["params"]["r"]
dt = res["dt"]
N = res["params"]["N"]

# Affichage des fonctions de continuation C_n, n=0,...,N-1
fig, axes = plt.subplots(2, 5, figsize=(18, 6), sharex=True, sharey=True)
axes = axes.ravel()

for k in range(N):
    Ck = continuation_on_grid(models, k, x_grid)
    axes[k].plot(x_grid, Ck, color="tab:blue", lw=2, label=fr"$\hat C_{k}(x)$")
    axes[k].set_title(f"k={k}")
    axes[k].grid(alpha=0.3)

axes[0].legend(loc="upper right", fontsize=9)
fig.suptitle("Fonctions de continuation apprises $\\hat C_k$ (Longstaff-Schwartz)", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Affichage des fonctions valeurs V_n, n=0,...,N
fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharex=True, sharey=True)
axes = axes.ravel()

for k in range(N + 1):
    Vk = value_on_grid(models, k, x_grid, K, r, dt)
    phik = payoff_discounted(x_grid, k, K, r, dt)

    axes[k].plot(x_grid, Vk, color="tab:green", lw=2, label=fr"$\hat V_{k}(x)$")
    axes[k].plot(x_grid, phik, color="tab:red", lw=1.5, ls="--", label=fr"$\phi_{k}(x)$")
    axes[k].set_title(f"k={k}")
    axes[k].grid(alpha=0.3)

# 12 cases pour 11 dates -> on masque la derniere case vide
axes[-1].axis("off")
axes[0].legend(loc="upper right", fontsize=8)
fig.suptitle("Fonctions valeurs estimees $\\hat V_k$ et payoff immediat $\\phi_k$", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Coefficients des regressions (utile pour verifier l'apprentissage numeriquement)
models = res["models"]
N = res["params"]["N"]

print("\nModeles de continuation appris:")
for k in range(N):
    mk = models[k]
    if mk["kind"] == "poly2":
        b0, b1, b2 = mk["beta"]
        print(f"k={k:2d}: C_k(x) ~ {b0:.6e} + {b1:.6e} x + {b2:.6e} x^2")
    else:
        print(f"k={k:2d}: C_k(x) ~ constante = {mk['c']:.6f}")

## Commentaire final (court)

- Le notebook implemente bien LS pour un put bermudeen aux dates \(k/N\).
- Les courbes \(\hat C_k\) (continuation) et \(\hat V_k\) (valeur) sont affichees **apres apprentissage**.
- La regle d'exercice est determinee par la comparaison locale
  \(\phi_k(X_k)\) vs \(\hat C_k(X_k)\), ce qui produit un temps d'arret empirique \(\hat\tau\) sur les trajectoires.

On peut augmenter `n_paths` pour stabiliser encore davantage les courbes si besoin.